[Course Home](../README.md) |
[Python Lab](../python/09_rate_distortion_and_information_bottleneck.ipynb) |
[Prev: Ch 08: Fisher Geometry](08_fisher_information_and_information_geometry.ipynb)
[Next: Ch 10: Active Learning & Experimental Design](10_active_learning_and_experimental_design.ipynb)

*Part V: Modern Frontiers: AI, Representation & Biology — Chapter 09 of 11*

---

# Chapter 9: Lossy Compression & Learning —
# Rate-Distortion & The Information Bottleneck
### *Shannon's Rate-Distortion Bound, Tishby's Bottleneck, and*
### *$\beta$-VAEs as Trade-off Engines*

---

## 1. What If You Cannot Afford to Keep Everything?

In Chapters 1 and 2, we studied the mathematics of **lossless compression**.
Claude Shannon's Source Coding Theorem established that a discrete message
source $X$ can be compressed without any loss of fidelity if and only if the
transmission rate exceeds the Shannon entropy:
$$
R \ge H(X) \quad \text{[bits/symbol]}
$$

If a computer program requires bit-for-bit reconstruction—such as compiling
source code, executing binary machine instructions, or executing encrypted
financial ledger transactions—lossless compression is mandatory.

However, across the vast majority of physical engineering domains—streaming
4K video, transmitting spoken audio, processing autonomous vehicle camera
streams, and training deep representation models—**lossless transmission is an**
**impossible and counterproductive luxury**:

1. **Continuous Signals Have Infinite Entropy**: Real physical phenomena
   (audio acoustic pressures, electromagnetic radar returns, photographic
   illumination) take values in continuous real spaces. As proved in Chapter 6,
   the differential entropy $h(X)$ does not represent discrete bit counts. To
   specify a single real number with infinite decimal precision requires
   infinitely many bits ($H(X_\delta) \to \infty$ as $\delta \to 0$). All
   digital recording of the continuous world is inherently lossy.
2. **Perceptual and Task Invariance**: Biological perceptual systems and
   downstream machine learning models do not care about microscopic photon
   noise or imperceptible high-frequency fluctuations. The human retina cannot
   resolve high-frequency chromatic variations (enabling chroma subsampling).
   A deep speech recognition model does not care about background room
   reverberation phase details; it only cares about extracting phonemes.

This fundamental reality poses a new mathematical question:
> *"If we are willing to tolerate an average distortion $D$ between the*
> *original signal $X$ and its reconstructed estimate $\hat{X}$, what is the*
> *absolute minimum bit-rate $R(D)$ required to transmit it?"*

In 1959, Claude Shannon established **Rate-Distortion Theory**, completing the
theoretical foundation of lossy data compression. Decades later, this theory
became the bedrock for understanding deep neural representations through
**Tishby's Information Bottleneck Principle** and **Variational Autoencoders**.

---

## 2. The Rate-Distortion Function $R(D)$

### Mathematical Formulation
Let $X \in \mathcal{X}$ be a random source symbol drawn from distribution
$P(X)$. We pass $X$ through an encoder and channel to produce a reconstructed
representation $\hat{X} \in \hat{\mathcal{X}}$.

To quantify the degradation between original and reconstructed symbols, we
define a non-negative **Distortion Metric**:
$$
d(x, \hat{x}) \ge 0 \quad \text{with } d(x, \hat{x}) = 0 \iff x = \hat{x}
$$

For a sequence of $n$ symbols, distortion is defined additively:
$$
d(x^n, \hat{x}^n) = \frac{1}{n} \sum_{i=1}^n d(x_i, \hat{x}_i)
$$

The expected distortion under a conditional reproduction channel
$P(\hat{X} \mid X)$ is:
$$
\mathbb{E}[d(X, \hat{X})]
= \sum_{x \in \mathcal{X}} \sum_{\hat{x} \in \hat{\mathcal{X}}}
  P(x) P(\hat{x} \mid x) d(x, \hat{x})
$$

### Definition of the Rate-Distortion Function
The **Rate-Distortion Function** $R(D)$ is defined as the minimum mutual
information between source $X$ and reconstruction $\hat{X}$ over all
conditional distributions $P(\hat{X} \mid X)$ satisfying the average
distortion constraint:
$$
R(D) = \min_{P(\hat{X} \mid X) : \mathbb{E}[d(X, \hat{X})] \le D} I(X; \hat{X})
$$

Shannon's Rate-Distortion Theorem proves that:
- **Achievability**: For any rate $R > R(D)$, there exists a block code of
  length $n$ operating at bit-rate $R$ that achieves average distortion
  $\le D$.
- **Converse**: No code operating at bit-rate $R < R(D)$ can achieve average
  distortion $\le D$.

### Common Distortion Metrics
1. **Hamming Distortion** (Discrete alphabets):
   $$
   d(x, \hat{x}) = \begin{cases}
     0 & \text{if } x = \hat{x} \\
     1 & \text{if } x \ne \hat{x}
   \end{cases}
   $$
   Used for symbol error rates, classification labels, and digital data.
2. **Squared Error Distortion (MSE)** (Continuous signals):
   $$
   d(x, \hat{x}) = (x - \hat{x})^2
   $$
   Used for audio processing, imaging, video compression, and sensor streams.
3. **Perceptual / Latent Feature Distortion**:
   $$
   d(x, \hat{x}) = \|\phi(x) - \phi(\hat{x})\|_2^2
   $$
   where $\phi(\cdot)$ extracts deep visual embeddings (e.g., LPIPS, VGG).

### Fundamental Properties of $R(D)$
- **Non-negativity**: $R(D) \ge 0$ for all $D \ge 0$.
- **Monotonicity**: $R(D)$ is a non-increasing function of $D$. Granting a
  larger distortion budget can never increase the required bit rate.
- **Convexity**: $R(D)$ is strictly convex in $D$. Mixing two coding strategies
  yields a point lying above or on the optimal curve.
- **Boundary Points**:
  - At zero distortion for discrete sources: $R(0) = H(X)$.
  - At maximum distortion
    $D_{\max} = \min_{\hat{x}} \mathbb{E}[d(X, \hat{x})]$:
    $R(D_{\max}) = 0$. The transmitter sends zero bits; the receiver guesses
    the unconditional optimal constant $\hat{x}^*$.

```
           THE RATE-DISTORTION CURVE R(D)
           
    Rate R(D) [bits]
       |
  H(X) *  (Lossless limit)
       | \
       |  \
       |   \  Feasible Operating Region
       |    \     (R >= R(D))
       |     *
       |      \
       |       \
     0 +--------*------------------------> Distortion D
       0       D_max
              (Zero bits needed: output unconditional mean)
```

### Derivation of $R(D)$ for a Gaussian Source under MSE
Consider a continuous zero-mean Gaussian source
$X \sim \mathcal{N}(0, \sigma^2)$ under squared error distortion
$d(x, \hat{x}) = (x - \hat{x})^2$ with budget
$\mathbb{E}[(X - \hat{X})^2] \le D$.

#### Step 1: Lower Bounding Mutual Information
By definition of mutual information for continuous variables:
$$
I(X; \hat{X}) = h(X) - h(X \mid \hat{X})
$$

Recognizing that subtracting a known quantity does not alter entropy:
$$
h(X \mid \hat{X}) = h(X - \hat{X} \mid \hat{X})
$$

Because conditioning can never increase differential entropy:
$$
h(X - \hat{X} \mid \hat{X}) \le h(X - \hat{X})
$$

The error term $e = X - \hat{X}$ has variance:
$$
\text{Var}(e) \le \mathbb{E}[(X - \hat{X})^2] \le D
$$

Among all continuous probability distributions with variance bounded by $D$,
the **Gaussian distribution uniquely maximizes differential entropy**
(as proved in Chapter 7):
$$
h(X - \hat{X}) \le \frac{1}{2} \log_2(2\pi e D)
$$

For the source $X \sim \mathcal{N}(0, \sigma^2)$, its differential entropy is:
$$
h(X) = \frac{1}{2} \log_2(2\pi e \sigma^2)
$$

Subtracting the two terms yields a rigorous lower bound:
$$
I(X; \hat{X}) \ge \frac{1}{2} \log_2(2\pi e \sigma^2)
- \frac{1}{2} \log_2(2\pi e D)
= \frac{1}{2} \log_2\left(\frac{\sigma^2}{D}\right)
$$

#### Step 2: Achieving the Lower Bound
Can this bound be achieved with equality? Yes.
Construct the backward test channel:
$$
X = \hat{X} + Z
$$
where $\hat{X} \sim \mathcal{N}(0, \sigma^2 - D)$ and $Z \sim \mathcal{N}(0, D)$
is independent additive Gaussian noise.

Under this construction:
- $\mathbb{E}[(X - \hat{X})^2] = \mathbb{E}[Z^2] = D$.
- $X - \hat{X} = Z$ is independent of $\hat{X}$, so
  $h(X - \hat{X} \mid \hat{X}) = h(Z) = \frac{1}{2}\log_2(2\pi e D)$.
- The mutual information matches the lower bound exactly!

#### Closed-Form Gaussian Rate-Distortion Function
$$
R(D) = \begin{cases}
\frac{1}{2} \log_2\left(\frac{\sigma^2}{D}\right)
  & \text{if } 0 < D \le \sigma^2 \\
0 & \text{if } D > \sigma^2
\end{cases}
$$

When $D \ge \sigma^2$, the allowed distortion equals or exceeds the total
signal variance. The encoder sends nothing ($R = 0$), and the decoder simply
outputs $\hat{X} = \mathbb{E}[X] = 0$.

### Reverse Water-Filling for Parallel Gaussian Sources
Suppose we have a vector of $m$ independent Gaussian sources:
$$
\mathbf{X} = (X_1, X_2, \dots, X_m), \quad X_i \sim \mathcal{N}(0, \sigma_i^2)
$$
with variances sorted in descending order:
$\sigma_1^2 \ge \sigma_2^2 \ge \dots \ge \sigma_m^2$.

We wish to compress all $m$ sources under a total distortion budget:
$$
\sum_{i=1}^m D_i \le D
$$

We must solve the constrained optimization problem:
$$
\min_{D_1, \dots, D_m} \sum_{i=1}^m
  \max\left(0, \frac{1}{2} \log_2 \frac{\sigma_i^2}{D_i}\right)
$$
$$
\text{subject to } \sum_{i=1}^m D_i \le D, \quad D_i > 0
$$

Applying the Karush-Kuhn-Tucker (KKT) conditions yields the famous
**Reverse Water-Filling** algorithm:
$$
D_i = \min(\theta, \sigma_i^2)
$$
$$
R_i = \max\left(0, \frac{1}{2} \log_2 \frac{\sigma_i^2}{\theta}\right)
$$
where the "water level" $\theta$ is chosen so that
$\sum_{i=1}^m \min(\theta, \sigma_i^2) = D$.

```
               REVERSE WATER-FILLING FOR PARALLEL SOURCES
               
      Variance sigma_i^2
           |
       s_1 | [  Bits R_1  ]
           |====================  <- Water level theta
           | [Distortion D_1]
       ----+---------------------
       s_2 | [  Bits R_2  ]
           |====================  <- Water level theta
           | [Distortion D_2]
       ----+---------------------
       s_3 |      (Empty)         <- sigma_3^2 <= theta:
           | [Distortion = s_3]      All variance discarded into D!
       ----+---------------------    Zero bits allocated (R_3 = 0).
              Source Component
```

Physical intuition of Reverse Water-Filling:
- **High-variance components** (strong signal power $\sigma_i^2 > \theta$)
  receive dedicated bit rate $R_i = \frac{1}{2} \log_2(\sigma_i^2 / \theta)$.
  Their distortion is capped at the uniform threshold $\theta$.
- **Low-variance components** (weak noise power $\sigma_i^2 \le \theta$)
  receive **zero bits** ($R_i = 0$). They are discarded entirely, contributing
  their full variance $\sigma_i^2$ to the distortion budget.

This is the exact mathematical principle behind JPEG transform coding (DCT),
MP3 subband coding, and Principal Component Analysis (PCA) compression.

---

## 3. Tishby's Information Bottleneck Principle (1999)

### Beyond Hand-Crafted Distortion Metrics
In classical Rate-Distortion theory, who decides what distortion metric
$d(x, \hat{x})$ to use?
Traditionally, an engineer chooses mean squared error or Hamming loss simply
because they are mathematically tractable.

However, in representation learning, machine perception, and biological
cognition, we compress data **for a specific downstream purpose**:
- An autonomous driving camera takes high-dimensional pixel inputs $X$.
- We need to predict target labels $Y$ (pedestrians, vehicles, signs).
- We do not care if cloud patterns in the sky are distorted. We only care
  about preserving features of $X$ that are informative about $Y$.

In 1999, physicist and computer scientist **Naftali Tishby**, along with
Fernando Pereira and William Bialek, asked:
> *"How can we extract a compressed internal representation $T$ of input $X$*
> *that preserves the maximum possible information about a target $Y$?"*

### The Information Bottleneck Triad
We formulate the system as a Markov chain:
$$
Y \longleftrightarrow X \longleftrightarrow T
$$
- $X$: Input data (image pixels, text documents, raw speech audio).
- $Y$: Target variable (class labels, future states, semantic concepts).
- $T$: Compressed representation (latent vector, hidden layer activations).

The Markov condition implies that $T$ is computed solely from $X$.
Given $X$, the representation $T$ is conditionally independent of $Y$:
$$
P(t, y \mid x) = P(t \mid x) P(y \mid x)
$$

### The Dual Objective
An optimal representation $T$ must resolve two conflicting forces:
1. **Compress the Input**: Strip away extraneous irrelevant noise:
   $$
   \min_{P(T \mid X)} I(X; T)
   $$
2. **Preserve the Target**: Retain all features predictive of the task:
   $$
   \max_{P(T \mid X)} I(T; Y)
   $$

### The Information Bottleneck Lagrangian
Combining these two objectives with a Lagrange multiplier $\beta$:
$$
\min_{P(T \mid X)} \mathcal{L}_{\text{IB}} = I(X; T) - \beta I(T; Y)
$$
where $\beta \in (0, \infty)$ is an inverse temperature parameter governing
the trade-off:
- $\beta \to 0$: Extreme compression. $T$ ignores $X$ completely
  ($I(X; T) = 0, I(T; Y) = 0$).
- $\beta \to \infty$: Maximum preservation. $T$ preserves all information
  present in $X$ that correlates with $Y$ ($I(T; Y) \to I(X; Y)$).

```
                 THE INFORMATION BOTTLENECK TRADE-OFF
                 
        Target Y <========== Input X -----------> Bottleneck T
                     I(X; Y)           I(X; T)
                        |                 |
                        |                 v
                        +===========> Preserve I(T; Y)
                        
     Objective: Minimize I(X; T) while Maximizing I(T; Y)
```

### Self-Consistent Equations and Intrinsic Distortion
By taking variational derivatives of $\mathcal{L}_{\text{IB}}$ with respect to
the stochastic mapping $P(t \mid x)$, Tishby et al. proved that the optimal
encoder satisfies the self-consistent equation:
$$
P(t \mid x) = \frac{P(t)}{Z(x, \beta)} \exp\left(
  -\beta D_{\text{KL}}\left(P(Y \mid x) \;\parallel\; P(Y \mid t)\right)
\right)
$$
where $Z(x, \beta)$ is a normalizing partition function:
$$
Z(x, \beta) = \sum_t P(t) \exp\left(
  -\beta D_{\text{KL}}\left(P(Y \mid x) \;\parallel\; P(Y \mid t)\right)
\right)
$$

Compare this formula to Shannon's classical rate-distortion equation:
$$
P(\hat{x} \mid x) = \frac{P(\hat{x})}{Z(x)} \exp(-\lambda d(x, \hat{x}))
$$

**The Grand Insight of the Information Bottleneck:**
The distortion metric is **not hand-crafted by human bias**!
The optimal distortion metric emerges naturally from information theory as:
$$
d_{\text{IB}}(x, t)
= D_{\text{KL}}\left(P(Y \mid x) \;\parallel\; P(Y \mid t)\right)
$$

The "cost" of assigning input $x$ to representation $t$ is the relative
entropy between what the raw input knows about target $Y$ and what the
compressed representation $t$ knows about target $Y$.

---

## 4. Deep Learning & Representation Learning Connection

### Deep Neural Networks as Successive Bottlenecks
A feedforward deep neural network with $L$ hidden layers processing input $X$
to predict target $Y$ forms a discrete Markov chain across its representations:
$$
X \longrightarrow H_1 \longrightarrow H_2 \longrightarrow
\dots \longrightarrow H_L \longrightarrow \hat{Y}
$$

Applying the **Data Processing Inequality** (Chapter 4):
$$
I(X; H_1) \ge I(X; H_2) \ge \dots \ge I(X; H_L) \ge I(X; \hat{Y})
$$
$$
I(Y; X) \ge I(Y; H_1) \ge I(Y; H_2) \ge \dots \ge I(Y; \hat{Y})
$$

Every successive hidden layer can only destroy or maintain information; it can
never create new information about either $X$ or $Y$.

### The Information Plane: Two Phases of Learning
In 2015, Naftali Tishby and Ravid Shwartz-Ziv tracked the trajectory of deep
networks in the **Information Plane** $(I(X; H), I(Y; H))$ during training:

```
    Target Info I(Y; H)
       |
  I(X; Y)*                     Phase 1: Fitting (ERM)
       |                      /  Weights adapt, I(X;H) and I(Y;H) grow
       |          * <------- *
       |         / (Phase 2)
       |        /
       |       / Phase 2: Compression (Generalization)
       |      /  SGD noise diffuses representations;
       |     *   irrelevant features compressed out.
     0 +-----+-------------------------------> Input Info I(X; H)
       0    I(T_opt; Y)
```

1. **Phase 1: Fitting (Empirical Risk Minimization)**: Early in training,
   gradients rapidly align network weights with the training labels. Both
   $I(X; H)$ and $I(Y; H)$ increase simultaneously as the layers extract
   features.
2. **Phase 2: Representation Compression**: Later in training, the stochastic
   noise inherent in mini-batch SGD acts like thermal diffusion. The network
   discards non-causal input features while maintaining task-relevant features.
   $I(X; H)$ contracts, moving representations toward the theoretical
   Information Bottleneck bound.

### Variational Autoencoders (VAEs) as Rate-Distortion Engines
In modern generative modeling, a Variational Autoencoder (Kingma & Welling 2013)
optimizes the **Evidence Lower Bound (ELBO)** on the marginal log-likelihood
of data:
$$
\ln p_\theta(x) \ge \mathcal{L}_{\text{ELBO}}(\theta, \phi)
= \mathbb{E}_{q_\phi(z \mid x)}[\ln p_\theta(x \mid z)]
- D_{\text{KL}}\left(q_\phi(z \mid x) \;\parallel\; p(z)\right)
$$

Under the lens of Information Theory, the ELBO is **literally Shannon's**
**Rate-Distortion objective**:

1. **Distortion Term**:
   $$
   D = -\mathbb{E}_{q_\phi(z \mid x)}[\ln p_\theta(x \mid z)]
   $$
   Under a Gaussian decoder
   $p_\theta(x \mid z) = \mathcal{N}(\mu(z), \sigma^2 I)$, this negative
   log-likelihood is proportional to Mean Squared Error:
   $$
   D \propto \mathbb{E}_{q_\phi(z \mid x)}[\|x - \mu(z)\|_2^2]
   $$
2. **Rate Term**:
   $$
   R = \mathbb{E}_{p_{\text{data}}(x)}\left[
     D_{\text{KL}}\left(q_\phi(z \mid x) \;\parallel\; p(z)\right)
   \right]
   $$
   By the Bit-Back Coding Theorem (Honkela & Valpola 2004), this KL divergence
   measures the average code length in nats required to transmit the latent
   representation $z$ across a channel using prior $p(z)$.

Maximizing the ELBO corresponds directly to minimizing the lossy
compression cost:
$$
\max \mathcal{L}_{\text{ELBO}} \iff \min [D + R]
$$

### The $\beta$-VAE Formulation (Higgins et al., 2017)
Irina Higgins et al. introduced the $\beta$-VAE by scaling the latent KL term:
$$
\mathcal{L}_{\beta\text{-VAE}}
= \mathbb{E}_{q_\phi(z \mid x)}[\ln p_\theta(x \mid z)]
- \beta D_{\text{KL}}\left(q_\phi(z \mid x) \;\parallel\; p(z)\right)
$$

This is the exact Lagrangian of the Rate-Distortion / Information Bottleneck:
$$
\min [D + \beta R]
$$

- **High Rate Penalty ($\beta > 1$)**: Forces the network through a narrow
  information bottleneck. The latent dimensions are penalized heavily for
  communicating bits, forcing coordinates to align with the true independent
  factors of variation in the world: **disentanglement** (e.g., lighting,
  object rotation, scale separated into orthogonal latent dimensions).
- **Low Rate Penalty ($\beta < 1$)**: The network prioritizes pixel-perfect
  reconstruction ($D \to 0$) at the cost of entangled, noisy latents.

---

## 5. Applied Engineering Contexts

### 1. Modern Video Codecs (H.264, HEVC / H.265, AV1)
Every modern video codec relies on **Rate-Distortion Optimization (RDO)** to
make compression decisions for every $16 \times 16$ or $64 \times 64$
macroblock:

For each block, the encoder must choose among dozens of potential coding modes
$m \in \mathcal{M}$ (intra-frame spatial directional prediction, inter-frame
motion vectors, block partitioning, transform skip).

The encoder evaluates the Lagrangian cost:
$$
J(m) = D(m) + \lambda_{\text{mode}} R(m)
$$
where $D(m)$ is the reconstruction distortion (SSE or SSIM) and $R(m)$ is the
number of bits required to encode the prediction mode, motion vectors, and
residual transform coefficients.

The Lagrange multiplier $\lambda$ is chosen directly from the slope of the
theoretical rate-distortion curve at the target Quantization Parameter (QP):
$$
\lambda = c \cdot 2^{\frac{\text{QP} - 12}{3}}
$$

### 2. Lossy Audio Compression (MP3, AAC, Opus)
Human auditory perception is non-linear and bandwidth-limited. The **Human**
**Auditory System (HAS)** exhibits powerful psychoacoustic masking effects:
- **Frequency Masking**: A loud tone at $1000\text{ Hz}$ drowns out softer
  tones at $1050\text{ Hz}$.
- **Temporal Masking**: A sudden transient sound (e.g., cymbal crash) masks
  sounds immediately preceding and following it for several milliseconds.

Audio codecs (MP3, AAC, Opus) partition audio into time-frequency subbands
using MDCT (Modified Discrete Cosine Transform). Rather than minimizing
raw Euclidean MSE, they apply Reverse Water-Filling to allocate bits such
that quantization distortion remains strictly below the **Sound Pressure**
**Masking Threshold** in every subband:
$$
\text{Distortion}(f) \le \text{Threshold}_{\text{masking}}(f)
$$

Bits are allocated only to spectral components where quantization noise
would cross the human audibility threshold.

### 3. Learned Neural Image Compression
Classical transform codecs (JPEG, JPEG-XL, WebP) use fixed linear transforms
(DCT, wavelets) hand-tuned by standards bodies over decades.

Modern learned neural compression (Johannes Ballé, David Minnen, et al.)
replaces hand-crafted transforms with end-to-end deep variational networks:
- **Non-linear Analysis Transform**: $y = g_a(x; \phi)$
- **Uniform Quantization**: $\hat{y} = \lfloor y \rceil$
- **Non-linear Synthesis Transform**: $\hat{x} = g_s(\hat{y}; \theta)$
- **Hyperprior Entropy Model**: A secondary variational network that estimates
  the conditional variance $\sigma_i^2$ of latent latents, optimizing
  lossless arithmetic coding rates.

The entire architecture is trained end-to-end via gradient descent on the
rate-distortion objective:
$$
\mathcal{L}_{\text{RD}} = R(\hat{y}) + \lambda D(x, \hat{x})
$$
Learned neural codecs currently exceed HEVC and VVC in perceptual visual
quality at identical bit rates.

### 4. Representation Pruning & Knowledge Distillation
In edge AI and autonomous hardware, multi-gigabyte models cannot run under
memory and power limits. The Information Bottleneck provides the guiding
framework for model compression:
- **Weight & Activation Pruning**: Discard channels and weights $W_i$ that
  contribute minimal mutual information $I(W_i; Y)$ relative to their memory
  footprint.
- **Knowledge Distillation**: A compact student network $S$ is trained to
  maximize the mutual information $I(T_S; T_T)$ with the internal bottleneck
  representations of a massive teacher network $T$.

---

[Course Home](../README.md) |
[Python Lab](../python/09_rate_distortion_and_information_bottleneck.ipynb) |
[Prev: Ch 08: Fisher Geometry](08_fisher_information_and_information_geometry.ipynb)
[Next: Ch 10: Active Learning & Experimental Design](10_active_learning_and_experimental_design.ipynb)

